### 01 — Data Completeness Overview

Cel: przegląd kompletności danych we wszystkich warstwach (bronze → silver → gold).  
Pytania: ile rekordów per tabela, jaki zakres dat, ile unikalnych kluczy, czy są puste tabele.  
Ostatnie uruchomienie: 26.03.2026

In [0]:
%sql
SELECT 'bronze.ohlcv_indicators' AS table_name,
COUNT(*) AS total_rows,
COUNT(DISTINCT symbol) AS unique_keys,
MIN(date) AS min_date,
MAX(date) AS max_date
FROM bronze.ohlcv_indicators

UNION ALL

SELECT 'bronze.fred_macro_indicators',
COUNT(*),
COUNT(DISTINCT indicator_id),
MIN(date),
MAX(date)
FROM bronze.fred_macro_indicators

UNION ALL

SELECT 'bronze.av_sentiment',
COUNT(*),
COUNT(DISTINCT symbol),
MIN(published_at),
MAX(published_at)
FROM bronze.av_sentiment

UNION ALL

SELECT 'bronze.fred_macro_metadata_indicators',
COUNT(*),
COUNT(DISTINCT indicator_id),
NULL,
NULL
FROM bronze.fred_macro_metadata_indicators

UNION ALL

SELECT 'bronze.qqq_etf_categories',
COUNT(*),
COUNT(DISTINCT symbol),
NULL,
NULL
FROM bronze.qqq_etf_categories

UNION ALL

SELECT 'bronze.qqq_etf_constituents',
COUNT(*),
COUNT(DISTINCT symbol),
NULL,
NULL
FROM bronze.qqq_etf_constituents

UNION ALL

SELECT 'silver.av_sentiment',
COUNT(*),
COUNT(DISTINCT symbol),
MIN(published_at),
MAX(published_at)
FROM silver.av_sentiment

UNION ALL

SELECT 'silver.fred_macro_indicators',
COUNT(*),
COUNT(DISTINCT indicator_id),
MIN(date),
MAX(date)
FROM silver.fred_macro_indicators

UNION ALL

SELECT 'silver.fred_macro_metadata_indicators',
COUNT(*),
COUNT(DISTINCT indicator_id),
NULL,
NULL
FROM silver.fred_macro_metadata_indicators

UNION ALL

SELECT 'silver.ohlcv_indicators',
COUNT(*),
COUNT(DISTINCT symbol),
MIN(date),
MAX(date)
FROM silver.ohlcv_indicators

UNION ALL

SELECT 'silver.qqq_entities',
COUNT(*),
COUNT(DISTINCT symbol),
NULL,
NULL
FROM silver.qqq_entities

UNION ALL

SELECT 'silver.qqq_etf_categories',
COUNT(*),
COUNT(DISTINCT symbol),
NULL,
NULL
FROM silver.qqq_etf_categories

UNION ALL

SELECT 'gold.av_sentiment_aggregated',
COUNT(*),
COUNT(DISTINCT symbol),
MIN(date),
MAX(date)
FROM gold.av_sentiment_aggregated

UNION ALL

SELECT 'gold.av_sentiment_sector_daily',
COUNT(*),
COUNT(DISTINCT industry),
MIN(date),
MAX(date)
FROM gold.av_sentiment_sector_daily

UNION ALL

SELECT 'gold.fred_with_dimension',
COUNT(*),
COUNT(DISTINCT indicator_id),
MIN(date),
MAX(date)
FROM gold.fred_with_dimension

UNION ALL

SELECT 'gold.macro_impact_on_tech',
COUNT(*),
NULL,
MIN(year_month),
MAX(year_month)
FROM gold.macro_impact_on_tech

UNION ALL

SELECT 'gold.ohlcv_with_dimension',
COUNT(*),
COUNT(DISTINCT symbol),
MIN(date),
MAX(date)
FROM gold.ohlcv_with_dimension

UNION ALL

SELECT 'gold.sentiment_lead_lag',
COUNT(*),
COUNT(DISTINCT symbol),
MIN(date),
MAX(date)
FROM gold.sentiment_lead_lag

UNION ALL

SELECT 'gold.sentiment_vs_returns',
COUNT(*),
COUNT(DISTINCT symbol),
MIN(date),
MAX(date)
FROM gold.sentiment_vs_returns

### Wnioski prawidłowe:
1. bronze.fred_macro_indicators - 
- total_rows ok, po drugim run pipeline overwrite dane tylko 12msc. refresh window.
- unique_keys ok, 8 series_ids zgodnie z symbolami w config
- MIN/MAX date ok.
2. silver.fred_macro_indicators - 
- total_rows ok, pełna historia przy pierwszym run pipeline.
- unique_keys ok, bronze 8 - silver 8.'
- MIN/MAX date ok, pełna historia 
3. gold.fred_with_dimension
- total_rows ok, spojne silver -> gold
- unique_keys ok
- MIN/MAX date ok
---
1. bronze.av_sentiment -
- total_rows ok, 
- unique_keys ok, extrakcja tylko dla sektoru technology 41 symboli na podstawie qqq entities.
- MIN/MAX date ok, inkrementacja danych
2. silver.av_sentiment -
- total_rows ok, deduplikacja danych
- unique_keys ok, zgodne bronze i silver
- MIN/MAX date ok, zgodne bronze i silver
3. gold.av_sentiment_aggregated -
- total_rows ok >wiersze po agregacji
- unique_keys ok spojne bronze -> silver -> gold
- MIN/MAX date spojne bronze -> silver -> gold
4. gold.av_sentiment_sector_daily
- total_rows ok mniej wierszy po agregacji
- unique_keys ok
- MIN/MAX date ok
---
1. bronze.fred_macro_metadata_indicators -
- total_rows ok
- unique_keys ok, zgodne z fred_macro_indicators 8 symboli
2. silver.fred_macro_metadata_indicators
- total rows ok zgodne z bronze
- unique_keys ok, zgodne z bronze
---
1. bronze.qqq_etf_categories
- total_rows ok 
- unique_keys ok
2. silver.qqq_etf_categories
- total_rows ok, 103 spojne miedzy bronze i silver
- unique_kesy ok, 103 spojne miedzy bronze i silver
---
1. bronze.qqq_etf_constituents
- total_rows ok
- unique_rows ok
2. silver.qqq_etf_constituents
- total_rows ok spojne bronze -> silver
- unique_rows ok spojnie bronze -> silver

### Problemy
1. ohlcv_indicators
  - bronze.ohlcv_indicators, brak danych bootstrap (Zmienic sposob zapisu danych w api_to_bronze z overwrite na upsert oraz dropnac tabele z bronze i zrobic rerun pipelinu)
  - silver.ohlcv_indicators, brak danych od 2k10 roku (naprawic bronze problem zniknie yolo)
  - gold. wszystkie tabele ktore sa zlaczone z ohlcv maja brakujace dane historyczne.
2. Tabele - 
(gold.macro_impact_on_tech, gold.ohlcv_with_dimension, gold.sentiment_vs_returns, gold.sentiment_lead_lag.) Braki danych od 2010 roku, spowodowane bledem w ohlcv api_to_bronze. Po naprawie ohlcv api_to_bronze powinny wrocic do normy po przeladowaniu historii i pełnym rerun pipelines.

In [0]:
%sql
SELECT COUNT(*) AS total,
SUM(CASE WHEN symbol IS NULL THEN 1 ELSE 0 END) AS null_symbol,
SUM(CASE WHEN date IS NULL THEN 1 ELSE 0 END) AS null_date,
SUM(CASE WHEN open IS NULL THEN 1 ELSE 0 END) AS null_open,
SUM(CASE WHEN high IS NULL THEN 1 ELSE 0 END) AS null_high,
SUM(CASE WHEN low IS NULL THEN 1 ELSE 0 END) AS null_low,
SUM(CASE WHEN close IS NULL THEN 1 ELSE 0 END) AS null_close,
SUM(CASE WHEN volume IS NULL THEN 1 ELSE 0 END) AS null_volume
FROM silver.ohlcv_indicators

In [0]:
%sql
SELECT COUNT(*) AS total,
SUM(CASE WHEN indicator_id IS NULL THEN 1 ELSE 0 END) AS null_indicator_id,
SUM(CASE WHEN date IS NULL THEN 1 ELSE 0 END) AS null_date,
SUM(CASE WHEN value IS NULL THEN 1 ELSE 0 END) AS null_value
FROM silver.fred_macro_indicators

In [0]:
%sql
SELECT indicator_id, COUNT(*) AS total, 
SUM(CASE WHEN value IS NULL THEN 1 ELSE 0 END) AS nulls 
FROM silver.fred_macro_indicators 
GROUP BY indicator_id 
ORDER BY nulls DESC

In [0]:
%sql
SELECT COUNT(*) AS total,
SUM(CASE WHEN symbol IS NULL THEN 1 ELSE 0 END) AS null_symbol,
SUM(CASE WHEN published_at IS NULL THEN 1 ELSE 0 END) AS null_published_at_,
SUM(CASE WHEN source IS NULL THEN 1 ELSE 0 END) AS null_source,
SUM(CASE WHEN title IS NULL THEN 1 ELSE 0 END) AS null_title,
SUM(CASE WHEN ticker_relevance_score IS NULL THEN 1 ELSE 0 END) AS null_ticker_relevance_score,
SUM(CASE WHEN ticker_sentiment_score IS NULL THEN 1 ELSE 0 END) AS null_ticker_sentiment_score,
SUM(CASE WHEN article_overall_sentiment_score IS NULL THEN 1 ELSE 0 END) AS null_article_overall_sentiment_score,
SUM(CASE WHEN article_overall_sentiment_label IS NULL THEN 1 ELSE 0 END) AS null_article_overall_sentiment_label,
SUM(CASE WHEN date IS NULL THEN 1 ELSE 0 END) AS null_date
FROM silver.av_sentiment

In [0]:
%sql
SELECT COUNT(*) AS total,
SUM(CASE WHEN symbol IS NULL THEN 1 ELSE 0 END) AS null_symbol,
SUM(CASE WHEN sector IS NULL THEN 1 ELSE 0 END) AS null_sector,
SUM(CASE WHEN industry IS NULL THEN 1 ELSE 0 END) AS null_industry
FROM silver.qqq_etf_categories

In [0]:
%sql
SELECT * FROM silver.qqq_etf_categories
WHERE symbol IS NULL OR industry IS NULL

### Null ratio — Silver
1. silver.ohlcv_indicators — 0 nulli we wszystkich kolumnach. OK.
2. silver.fred_macro_indicators — 1517 nulli w value (3.7%). Daily indykatory (dgs10, dgs2, t10yie) — weekendy/święta. Monthly (unrate, cpiaucsl, cpilfesl) — po 1 nullu, opóźnienie publikacji. Oczekiwane.
3. silver.av_sentiment — 0 nulli we wszystkich kolumnach. OK.
4. silver.qqq_etf_categories — 1 null w sector i industry. Symbol USD (cash position ETF). Oczekiwane.

In [0]:
%sql
SELECT COUNT(*) AS total,
SUM(CASE WHEN symbol IS NULL THEN 1 ELSE 0 END) AS null_symbol,
SUM(CASE WHEN date IS NULL THEN 1 ELSE 0 END) AS null_date,
SUM(CASE WHEN close IS NULL THEN 1 ELSE 0 END) AS null_close,
SUM(CASE WHEN daily_return IS NULL THEN 1 ELSE 0 END) AS null_daily_return,
SUM(CASE WHEN avg_sentiment_score IS NULL THEN 1 ELSE 0 END) AS null_avg_sentiment_score,
SUM(CASE WHEN article_count IS NULL THEN 1 ELSE 0 END) AS null_article_count
FROM gold.sentiment_vs_returns

In [0]:
%sql
SELECT COUNT(*) AS total,
SUM(CASE WHEN symbol IS NULL THEN 1 ELSE 0 END) AS null_symbol,
SUM(CASE WHEN date IS NULL THEN 1 ELSE 0 END) AS null_date,
SUM(CASE WHEN close IS NULL THEN 1 ELSE 0 END) AS null_close,
SUM(CASE WHEN return_t1 IS NULL THEN 1 ELSE 0 END) AS null_return_t1,
SUM(CASE WHEN return_t2 IS NULL THEN 1 ELSE 0 END) AS null_return_t2,
SUM(CASE WHEN return_t5 IS NULL THEN 1 ELSE 0 END) AS null_return_t5,
SUM(CASE WHEN article_count IS NULL THEN 1 ELSE 0 END) AS null_article_count,
SUM(CASE WHEN avg_sentiment_score IS NULL THEN 1 ELSE 0 END) AS null_avg_sentiment_score,
SUM(CASE WHEN avg_relevance_score IS NULL THEN 1 ELSE 0 END) AS null_avg_relevance_score,
SUM(CASE WHEN bullish_count IS NULL THEN 1 ELSE 0 END) AS null_bullish_count,
SUM(CASE WHEN somewhat_bullish_count IS NULL THEN 1 ELSE 0 END) AS null_somewhat_bullish_count,
SUM(CASE WHEN neutral_count IS NULL THEN 1 ELSE 0 END) AS null_neutral_count,
SUM(CASE WHEN somewhat_bearish_count IS NULL THEN 1 ELSE 0 END) AS null_somewhat_bearish_count,
SUM(CASE WHEN bearish_count IS NULL THEN 1 ELSE 0 END) AS null_bearish_count,
SUM(CASE WHEN source_api IS NULL THEN 1 ELSE 0 END) AS null_source_api
FROM gold.sentiment_lead_lag

In [0]:
%sql
SELECT COUNT(*),
SUM(CASE WHEN year_month IS NULL THEN 1 ELSE 0 END) AS null_year_month,
SUM(CASE WHEN monthly_return IS NULL THEN 1 ELSE 0 END) AS null_monthly_return,
SUM(CASE WHEN cpiaucsl IS NULL THEN 1 ELSE 0 END) AS null_cpiaucsl,
SUM(CASE WHEN cpilfesl IS NULL THEN 1 ELSE 0 END) AS null_cpilfesl,
SUM(CASE WHEN fedfunds IS NULL THEN 1 ELSE 0 END) AS null_fedfunds,
SUM(CASE WHEN dgs10 IS NULL THEN 1 ELSE 0 END) AS null_dgs10,
SUM(CASE WHEN dgs2 IS NULL THEN 1 ELSE 0 END) AS null_dgs2,
SUM(CASE WHEN unrate IS NULL THEN 1 ELSE 0 END) AS null_unrate,
SUM(CASE WHEN indpro IS NULL THEN 1 ELSE 0 END) AS null_indpro,
SUM(CASE WHEN t10yie IS NULL THEN 1 ELSE 0 END) AS null_t10yie
FROM gold.macro_impact_on_tech

### Null ratio — Gold
1. gold.sentiment_vs_returns — 0 nulli. OK.
2. gold.sentiment_lead_lag — 0 nulli. OK.
3. gold.ohlcv_with_dimension — 0 nulli. OK.
4. gold.fred_with_dimension — 0 nulli. OK.
5. gold.av_sentiment_aggregated — 0 nulli. OK.
6. gold.av_sentiment_sector_daily — 0 nulli. OK.
7. gold.macro_impact_on_tech — nulle w kolumnach makro (cpiaucsl: 2, cpilfesl: 2, fedfunds: 1, unrate: 2, indpro: 1). Opóźnienia publikacji indykatorów. Oczekiwane, po przeładowaniu historii OHLCV proporcja będzie marginalna.